In [3]:
import numpy as np
import seaborn as sb
import pandas as pd

In [4]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
tic_tac_toe_endgame = fetch_ucirepo(id=101) 
  
# data (as pandas dataframes) 
X = tic_tac_toe_endgame.data.features 
y = tic_tac_toe_endgame.data.targets 

from RuleTree.utils.feature_utils import detect_categorical_features
from RuleTree.stumps.classification.DecisionTreeStumpClassifier import DecisionTreeStumpClassifier
from RuleTree.stumps.classification.MofNTrepanStumpClassifier import MofNTrepanStumpClassifier
from RuleTree.base.RuleTreeBaseStump import RuleTreeBaseStump 
categorical_features = detect_categorical_features(X)

In [5]:
X.shape

(958, 9)

In [6]:
X.isnull().sum()

top-left-square         0
top-middle-square       0
top-right-square        0
middle-left-square      0
middle-middle-square    0
middle-right-square     0
bottom-left-square      0
bottom-middle-square    0
bottom-right-square     0
dtype: int64

In [7]:
X.info()
attributes = X.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   top-left-square       958 non-null    object
 1   top-middle-square     958 non-null    object
 2   top-right-square      958 non-null    object
 3   middle-left-square    958 non-null    object
 4   middle-middle-square  958 non-null    object
 5   middle-right-square   958 non-null    object
 6   bottom-left-square    958 non-null    object
 7   bottom-middle-square  958 non-null    object
 8   bottom-right-square   958 non-null    object
dtypes: object(9)
memory usage: 67.5+ KB


In [8]:
for col in X.columns:
    print(X[col].unique())

['x' 'o' 'b']
['x' 'o' 'b']
['x' 'o' 'b']
['x' 'o' 'b']
['o' 'b' 'x']
['o' 'b' 'x']
['x' 'o' 'b']
['o' 'x' 'b']
['o' 'x' 'b']


In [9]:
X.head()

,top-left-square,top-middle-square,top-right-square,middle-left-square,middle-middle-square,middle-right-square,bottom-left-square,bottom-middle-square,bottom-right-square
0,x,x,x,x,o,o,x,o,o
1,x,x,x,x,o,o,o,x,o
2,x,x,x,x,o,o,o,o,x
3,x,x,x,x,o,o,o,b,b
4,x,x,x,x,o,o,b,o,b


In [10]:
# Dizionario esteso per gestire anche i codici di ambiguità (sia maiuscoli che minuscoli per sicurezza)
extended_nucleotide_map = {
    'x': 1, 'X': 1,  # X
    'o': -1, 'O': -1,  # O
    'b': 0, 'B': 0   # Blank
}

X = X.replace(extended_nucleotide_map)
X.head()



C:\Users\david\AppData\Local\Temp\ipykernel_13452\2053523666.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.replace(extended_nucleotide_map)


,top-left-square,top-middle-square,top-right-square,middle-left-square,middle-middle-square,middle-right-square,bottom-left-square,bottom-middle-square,bottom-right-square
0,1,1,1,1,-1,-1,1,-1,-1
1,1,1,1,1,-1,-1,-1,1,-1
2,1,1,1,1,-1,-1,-1,-1,1
3,1,1,1,1,-1,-1,-1,0,0
4,1,1,1,1,-1,-1,0,-1,0


In [11]:
# 2. Estrai direttamente la lista dei nuovi attributi
attributes = list(X.columns)

# 3. Estrai la matrice dei valori per addestrare la rete e l'albero
X = X.values

In [12]:
y = y.replace({'negative': 0, 'positive' : 1})


C:\Users\david\AppData\Local\Temp\ipykernel_13452\2336503015.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = y.replace({'negative': 0, 'positive' : 1})


In [13]:
y.head()

,class
0,1
1,1
2,1
3,1
4,1


In [14]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
# Addestra una rete (come nel paper)
mlp = MLPClassifier(
    hidden_layer_sizes=(32),  # prova 0,5,10,20,40
    max_iter=500,
    random_state=42,
    learning_rate_init=0.1,
    activation='relu',
)
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)

c:\Users\david\miniconda3\envs\trepan-dev\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:1219: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [17]:
print('Test Accuracy %s' % accuracy_score(y_test, y_pred))
print('Test F1-score %s' % f1_score(y_test, y_pred, average=None))
print(classification_report(y_test, y_pred))

Test Accuracy 0.9947916666666666
Test F1-score [0.9924812  0.99601594]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99        67
           1       0.99      1.00      1.00       125

    accuracy                           0.99       192
   macro avg       1.00      0.99      0.99       192
weighted avg       0.99      0.99      0.99       192



In [18]:
from RuleTree.stumps.classification.MofNTrepanStumpClassifier import MofNTrepanStumpClassifier

In [19]:
# importa la classe direttamente dal file
base_stumps = [MofNTrepanStumpClassifier(max_conditions=9)]
from RuleTree.tree.TrepanClassifier import TrepanClassifier
trepan_clf = TrepanClassifier(
    estimator=mlp,
    stump_selection='best',
    s_min=2000,
    epsilon=0.1,
    delta=0.1,
    base_stumps=base_stumps,
    max_internal_nodes=15,
    categorical_features=categorical_features,
    random_state=42,
)
trepan_clf.fit(X_train, y_train)
y_pred_trepan = trepan_clf.predict(X_test)
print('Accuracy %s' % accuracy_score(y_test, y_pred_trepan))
print('F1-score %s' % f1_score(y_test, y_pred_trepan, average=None))
print(classification_report(y_test, y_pred_trepan))
fidelty = accuracy_score(y_pred, y_pred_trepan)
print("The fidelty respect to the oracle is %s" % fidelty)

Accuracy 0.8958333333333334
F1-score [0.83333333 0.92424242]
              precision    recall  f1-score   support

           0       0.94      0.75      0.83        67
           1       0.88      0.98      0.92       125

    accuracy                           0.90       192
   macro avg       0.91      0.86      0.88       192
weighted avg       0.90      0.90      0.89       192

The fidelty respect to the oracle is 0.9010416666666666


In [20]:
trepan_clf.print_trepan_rules(feature_names=attributes)


  REGOLE GLOBALI ESTRATTE DA TREPAN
REGOLA 1 [Nodo ID: Rl]:
  IF  (3-of-{ middle-middle-square == -1, top-middle-square == -1, bottom-middle-square == -1 }})
  THEN Predizione = 0
  [Fedeltà: 100.00%, Copertura: 3.92%]

REGOLA 2 [Nodo ID: Rrl]:
  IF  (NOT (3-of-{ middle-middle-square == -1, top-middle-square == -1, bottom-middle-square == -1 }})) AND 
      (3-of-{ bottom-right-square == -1, top-right-square == -1, middle-right-square == -1 }})
  THEN Predizione = 0
  [Fedeltà: 63.00%, Copertura: 3.52%]

REGOLA 3 [Nodo ID: Rrrll]:
  IF  (NOT (3-of-{ middle-middle-square == -1, top-middle-square == -1, bottom-middle-square == -1 }})) AND 
      (NOT (3-of-{ bottom-right-square == -1, top-right-square == -1, middle-right-square == -1 }})) AND 
      (1-of-{ middle-middle-square == -1 }}) AND 
      (3-of-{ bottom-left-square == 1, top-left-square == 1, middle-left-square == 1 }})
  THEN Predizione = 1
  [Fedeltà: 100.00%, Copertura: 3.92%]

REGOLA 4 [Nodo ID: Rrrlrll]:
  IF  (NOT (3-of-

In [21]:
def create_mlp(hidden_units):
    if hidden_units == 0:
        return MLPClassifier(
            hidden_layer_sizes=(),
            max_iter=500,  # aumentato
            random_state=42,
            early_stopping=False,
            learning_rate_init= 0.1
        )
    else:
        return MLPClassifier(
            hidden_layer_sizes=(hidden_units,),
            max_iter=500,
            random_state=42,
            early_stopping=False,
            learning_rate_init=0.1
        )

def select_best_hidden_units(X_train, y_train, hidden_units_list, cv_inner=5):
    """
    Seleziona il miglior numero di hidden unit con cross-validation interna.
    Come nel paper: prova e sceglie il migliore.
    """
    X_train = np.asarray(X_train, dtype=np.float64)
    # y_train è già int, non convertire
    
    best_units = 10
    best_score = -1
    
    for units in hidden_units_list:
        mlp = create_mlp(units)
        scores = cross_val_score(mlp, X_train, y_train, cv=cv_inner, scoring='accuracy')
        mean_score = np.mean(scores)
        
        if mean_score > best_score:
            best_score = mean_score
            best_units = units
            
    return best_units, best_score

def train_and_evaluate_network(X_train, y_train, X_test, y_test, hidden_units):
    """
    Addestra una rete neurale e restituisce:
    - accuracy sul test set
    - modello addestrato
    - predizioni sul test set
    """
    mlp = create_mlp(hidden_units)
    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc, mlp, y_pred

In [22]:
X_np = X
y_np = y.values 
# 10-FOLD CROSS-VALIDATION

hidden_units_list = [64,32,80]
n_folds = 10
random_state = 42

accuracies_net = []
accuracies_tree = []
fidelities = []
chosen_units = []

skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)

print("\n" + "="*60)
print("INIZIO 10-FOLD CV")
print("="*60 + "\n")

for fold, (train_idx, test_idx) in enumerate(skf.split(X_np, y_np), 1):
    print(f"Fold {fold}/{n_folds}")
    
    # 1. Dividi i dati (USANDO NUMPY!)
    X_train = X_np[train_idx]
    y_train = y_np[train_idx]
    X_test = X_np[test_idx]
    y_test = y_np[test_idx]
    
    
    # 3. Seleziona hidden units
    best_units, best_cv_score = select_best_hidden_units(
        X_train, y_train, hidden_units_list, cv_inner=5
    )
    chosen_units.append(best_units)
    print(f"  → Migliori hidden units: {best_units} (CV score interno: {best_cv_score:.3f})")
    
    # 4. Addestra rete
    net_acc, network, y_pred_net = train_and_evaluate_network(
        X_train, y_train, X_test, y_test, best_units
    )
    accuracies_net.append(net_acc)
    print(f"  → Accuratezza rete: {net_acc:.3f}")
    
    trepan_clf = TrepanClassifier(
    estimator=mlp,
    categorical_features=categorical_features,
    s_min=500,
    max_internal_nodes=15,
    random_state=42,
    base_stumps=base_stumps,
    )
    
    trepan_clf.fit(X_train, y_train)
    y_pred_trepan = trepan_clf.predict(X_test)  # usa X_test_scaled!
    
    tree_acc = accuracy_score(y_test, y_pred_trepan)
    fidelity = accuracy_score(y_pred_net, y_pred_trepan)
    
    accuracies_tree.append(tree_acc)
    fidelities.append(fidelity)
    
    print(f"  → Accuratezza albero: {tree_acc:.3f}")
    print(f"  → Fedeltà albero-rete: {fidelity:.3f}")
    print()

print("="*60)
print("RISULTATI FINALI (10-fold CV)")
print("="*60)
print(f"Accuratezza rete:     {np.mean(accuracies_net):.3f} ± {np.std(accuracies_net):.3f}")
print(f"Accuratezza albero:   {np.mean(accuracies_tree):.3f} ± {np.std(accuracies_tree):.3f}")
print(f"Fedeltà:              {np.mean(fidelities):.3f} ± {np.std(fidelities):.3f}")
print(f"Hidden units più scelte: {pd.Series(chosen_units).value_counts().to_dict()}")


INIZIO 10-FOLD CV

Fold 1/10
  → Migliori hidden units: 32 (CV score interno: 0.942)
  → Accuratezza rete: 1.000
  → Accuratezza albero: 0.875
  → Fedeltà albero-rete: 0.875

Fold 2/10
  → Migliori hidden units: 32 (CV score interno: 0.948)
  → Accuratezza rete: 0.979
  → Accuratezza albero: 0.927
  → Fedeltà albero-rete: 0.948

Fold 3/10
  → Migliori hidden units: 64 (CV score interno: 0.941)
  → Accuratezza rete: 1.000
  → Accuratezza albero: 0.896
  → Fedeltà albero-rete: 0.896

Fold 4/10
  → Migliori hidden units: 80 (CV score interno: 0.940)
  → Accuratezza rete: 1.000
  → Accuratezza albero: 0.875
  → Fedeltà albero-rete: 0.875

Fold 5/10
  → Migliori hidden units: 80 (CV score interno: 0.949)
  → Accuratezza rete: 0.979
  → Accuratezza albero: 0.938
  → Fedeltà albero-rete: 0.938

Fold 6/10
  → Migliori hidden units: 32 (CV score interno: 0.940)
  → Accuratezza rete: 0.990
  → Accuratezza albero: 0.896
  → Fedeltà albero-rete: 0.885

Fold 7/10
  → Migliori hidden units: 80 (CV 

In [23]:
import numpy as np
from sklearn.metrics import accuracy_score
from RuleTree.tree.TrepanClassifier import TrepanClassifier

# 1. Definisci i seed per lo stress test
seeds_da_testare = [10, 42, 111, 123, 999, 11, 86, 78, 123, 22, 52, 66, 88, 101, 222]

# 2. Inizializza le liste per raccogliere le metriche
varianza_accuracy = []
varianza_fidelity = []
varianza_foglie = []

print("Calcolo delle predizioni dell'oracolo (fisse per tutti i seed)...")
# L'oracolo è il modello mlp già addestrato
y_pred_oracolo = mlp.predict(X_test)

for seed in seeds_da_testare:
    print(f"\nAddestramento albero con Seed: {seed}")
    
    # Inizializza TrepanClassifier con il seed corrente
    trepan_clf = TrepanClassifier(
        estimator=mlp,
        categorical_features=categorical_features,
        s_min=500,
        max_internal_nodes=32,
        random_state=seed,
        base_stumps=[MofNTrepanStumpClassifier(max_conditions=3)],
        delta=0.1,
        epsilon=0.1
        )
    
    # Esegui il fit sui dati di training
    trepan_clf.fit(X_train, y_train)
    
    # Ottieni le predizioni dell'albero su X_test_scaled
    y_pred_trepan = trepan_clf.predict(X_test)
    
    # Calcola l'Accuracy
    acc = accuracy_score(y_test, y_pred_trepan)
    
    # Calcola la Fidelity
    fid = accuracy_score(y_pred_oracolo, y_pred_trepan)
    
    # Calcola il numero di foglie generate
    foglie = len(trepan_clf.get_leaf_nodes())
    
    # Aggiungi i valori alle liste
    varianza_accuracy.append(acc)
    varianza_fidelity.append(fid)
    varianza_foglie.append(foglie)

print("\n" + "="*50)
print(" RISULTATI TEST DI VARIANZA DEI RANDOM SEED")
print("="*50)

print(f"Accuracy : Media {np.mean(varianza_accuracy):.4f} ± Std {np.std(varianza_accuracy):.4f}")
print(f"Fidelity : Media {np.mean(varianza_fidelity):.4f} ± Std {np.std(varianza_fidelity):.4f}")
print(f"Foglie   : Media {np.mean(varianza_foglie):.2f} ± Std {np.std(varianza_foglie):.2f}")

Calcolo delle predizioni dell'oracolo (fisse per tutti i seed)...

Addestramento albero con Seed: 10

Addestramento albero con Seed: 42

Addestramento albero con Seed: 111

Addestramento albero con Seed: 123

Addestramento albero con Seed: 999

Addestramento albero con Seed: 11

Addestramento albero con Seed: 86

Addestramento albero con Seed: 78

Addestramento albero con Seed: 123

Addestramento albero con Seed: 22

Addestramento albero con Seed: 52

Addestramento albero con Seed: 66

Addestramento albero con Seed: 88

Addestramento albero con Seed: 101

Addestramento albero con Seed: 222

 RISULTATI TEST DI VARIANZA DEI RANDOM SEED
Accuracy : Media 0.8618 ± Std 0.0319
Fidelity : Media 0.8618 ± Std 0.0319
Foglie   : Media 32.87 ± Std 0.34
